# Econ 521 — Lab 2
## Analyzing and Designing a Randomized Controlled Trial


Last week we established what randomization buys you. This week we do the work: take a real
experiment apart, get the standard errors right, test the null without assuming a distribution
at all, and then run the calculation you should have done *before* collecting any data.

**Running example.** Thornton, Rebecca L. (2008), "The Demand for, and Impact of, Learning HIV
Status," *American Economic Review* 98(5): 1829–1863. In rural Malawi, respondents were tested
for HIV and then randomly offered a cash incentive to collect their results at a nearby voluntary
counseling and testing (VCT) center. The incentive was randomized at the individual level;
distance to the center was randomized at the village level. Outcome: whether they came and got
their results.


In [35]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.stats.power as smp
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
rng = np.random.default_rng(20260911)

from causaldata import thornton_hiv, ri

---
## §1 The data, attrition, and balance

**Variables.** `got` = collected HIV results (outcome); `any` = offered any cash incentive
(treatment); `tinc` = value of the incentive in dollars; `distvct` = distance to the VCT center
in km; `villnum` = village; `age`; `hiv2004` = HIV positive at the 2004 test.

First move on any dataset, always: look at what is missing and why.

In [36]:
thornton = thornton_hiv.load_pandas().data
print(thornton.shape)
print("\nmissing values by column:")
print(thornton.isna().sum().to_string())
thornton.describe().T

(4820, 7)

missing values by column:
villnum      27
got        1926
distvct       0
tinc       1919
any        1919
age         441
hiv2004    1926


,count,mean,std,min,25%,50%,75%,max
villnum,"4,793.0000",63.4776,48.3118,1.0000,14.0000,57.0000,111.0000,145.0000
got,"2,894.0000",0.6966,0.4598,0.0000,0.0000,1.0000,1.0000,1.0000
distvct,"4,820.0000",2.0026,1.2565,0.0000,1.0299,1.6748,2.7429,5.1916
tinc,"2,901.0000",0.9880,0.9014,0.0000,0.0946,0.9456,1.8912,2.8368
any,"2,901.0000",0.7659,0.4235,0.0000,1.0000,1.0000,1.0000,1.0000
age,"4,379.0000",33.6517,13.1629,11.0000,23.0000,32.0000,43.0000,84.0000
hiv2004,"2,894.0000",0.0591,0.2555,-1.0000,0.0000,0.0000,0.0000,1.0000


Two things should bother you.

**First, the missing-value pattern is not random across columns.** Roughly 1,900 rows are missing
`got`, `any`, `tinc`, and `hiv2004` simultaneously. Those are people who were surveyed but never
tested in 2004, so they were never eligible for the incentive experiment. They are not attrition
from the experiment; they were never in it. Dropping them is a decision about the *population*
your estimate describes, not a data-cleaning nicety, and you should say so in the paper.

**Second, look at the minimum of `hiv2004`.** It is $-1$. HIV status is supposed to be a 0/1
indicator. A value outside the logical range is almost always a missing-data sentinel that
somebody forgot to convert, and if you leave it in, it will silently enter every mean, every
regression, and every balance table you produce. Find these before they find you.

### Exercise 1 (6 min)

Build the analysis sample and a summary table.

1. Keep rows with non-missing `got`, `any`, `age`, `distvct`, and `hiv2004`.
2. Drop the $-1$ codes in `hiv2004`.
3. Produce a table with the **mean, standard deviation, min, and max** of `got`, `any`, `tinc`,
   `age`, `hiv2004`, `distvct`, and report $n$ and the number of villages underneath it.
4. Then produce a **balance table**: mean of `age`, `hiv2004`, `distvct` by treatment status,
   the difference, and a $p$-value for the difference.

Before you look at step 4: what do you *expect* the balance table to show, and what would you
conclude if `distvct` were imbalanced?

In [37]:
# ---- EXERCISE 1 -------------------------------------------------------------
d = thornton.dropna(subset=["got", "any", "age", "distvct", "hiv2004"]).copy()
d = d.loc[d["hiv2004"] != -1].copy()        # drop the -1 sentinel in hiv2004

summary_vars = ["got", "any", "tinc", "age", "hiv2004", "distvct"]
summary = d[summary_vars].agg(["mean", "std", "min", "max"]).T  # mean / std / min / max for summary_vars

print(summary)
print(f"\nN = {len(d):,} respondents in {d.villnum.nunique()} villages")
# -----------------------------------------------------------------------------


           mean     std     min     max
got      0.6911  0.4621  0.0000  1.0000
any      0.7805  0.4140  0.0000  1.0000
tinc     1.0065  0.8987  0.0000  2.8368
age     33.3704 13.6479 11.0000 80.0000
hiv2004  0.0629  0.2427  0.0000  1.0000
distvct  2.0128  1.2668  0.0000  5.1916

N = 2,816 respondents in 119 villages


In [38]:
# ---- EXERCISE 1, part 4: balance table --------------------------------------
def balance_row(data, var, treat="any"):
    '''Mean by arm, the difference, and a p-value for the difference.'''
    m = smf.ols(f"{var} ~ {treat}", data=data).fit(cov_type="HC1")
    return {
        "control mean": data.loc[data[treat] == 0, var].mean(),
        "treated mean": data.loc[data[treat] == 1, var].mean(),
        "difference":   m.params[treat],
        "std. error":   m.bse[treat],
        "p-value":      m.pvalues[treat],
    }

balance = pd.DataFrame(
    {var: balance_row(d, var) for var in ["age", "hiv2004", "distvct"]}
).T   # apply balance_row to age, hiv2004, distvct and stack into a DataFrame
balance
# -----------------------------------------------------------------------------


,control mean,treated mean,difference,std. error,p-value
age,32.1084,33.7252,1.6168,0.5939,0.0065
hiv2004,0.0631,0.0628,-0.0003,0.0111,0.9768
distvct,1.9513,2.0301,0.0788,0.0561,0.1599


**What you should find:** $n = 2{,}816$ in 119 villages. `hiv2004` and `distvct` are balanced.
`age` is **not** — the incentive group is about 1.6 years older, with $p \approx 0.007$.

Do not skip past that. It is the most useful thing in the table, and here is how to think about
it. The imbalance is 1.6 years on a mean of 33, roughly a tenth of a standard deviation. It is
detectable because $n$ is large, not because it is large. So the question is not "did we cross
0.05" but **"could a 1.6-year age gap plausibly generate a 45 percentage point difference in
collecting HIV results?"** Almost certainly not — and you can check by putting age in the
regression in §3 and watching the coefficient not move.

Two further caveats worth stating out loud. First, we are working with a *subsample*: we dropped
everyone missing any of five variables, and a subsample of a balanced sample need not be
balanced. If dropping observations creates imbalance, that is a differential attrition problem,
not a randomization problem, and it deserves its own paragraph in your paper. Second, recall from
last week that testing three covariates gives you about a 14% chance of one false rejection.
Report the balance table, discuss what is imbalanced, show it does not drive the result. Do not
either hide it or panic about it.

`distvct` is also worth flagging: it was randomized *at the village level*, which will matter
again in §5.

---
## §2 Difference in means = OLS, and the right standard error

Under randomization, $E[Y \mid D=1] - E[Y \mid D=0] = \text{ATE}$, so estimating the ATE is just
comparing two means. Three routes to the same number:

In [39]:
treated = d.loc[d["any"] == 1, "got"]
control = d.loc[d["any"] == 0, "got"]

print(f"1. Raw difference in means : {treated.mean() - control.mean():.4f}")

welch = stats.ttest_ind(treated, control, equal_var=False)
print(f"2. Welch t-test            : t = {welch.statistic:.3f}")

ols = smf.ols("got ~ any", data=d).fit(cov_type="HC1")
print(f"3. OLS with HC1 errors     : b = {ols.params['any']:.4f}, t = {ols.tvalues['any']:.3f}")
print(f"\ncontrol mean = {control.mean():.3f}")
ols.summary().tables[1]

1. Raw difference in means : 0.4479
2. Welch t-test            : t = 21.351
3. OLS with HC1 errors     : b = 0.4479, t = 21.359

control mean = 0.341


,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,0.3414,0.019,17.893,0.000,0.304,0.379
any,0.4479,0.021,21.359,0.000,0.407,0.489


The Welch $t$ and the heteroskedasticity-robust OLS $t$ agree to three decimals. That is not a
coincidence: allowing unequal variances across arms *is* what HC1 does in this regression. If
someone asks you whether to use a $t$-test or a regression, the honest answer is that the choice
does not exist — but the regression generalizes, so use the regression.

**Substance:** offering any cash incentive raised the probability of collecting HIV results from
about **34%** to about **79%**, an effect of roughly **45 percentage points**. This is a very
large effect for a very small amount of money, and it is the paper's headline: demand for
information about your own HIV status is extremely sensitive to trivial transaction costs.

### Now the standard errors

The rule is: **cluster at the level at which treatment was assigned.** Incentives were randomized
to individuals, so individual-level robust errors are defensible. But outcomes within a village
are correlated — people share a VCT center, a road, a rainy season — so let us see what
clustering does anyway.

In [40]:
fits = {
    "Classical (iid)": smf.ols("got ~ any", data=d).fit(),
    "Robust (HC1)":    smf.ols("got ~ any", data=d).fit(cov_type="HC1"),
    "Clustered by village": smf.ols("got ~ any", data=d).fit(
        cov_type="cluster", cov_kwds={"groups": d.villnum}),
}
pd.DataFrame({k: {"coef": f.params["any"], "std. error": f.bse["any"],
                  "t": f.tvalues["any"]} for k, f in fits.items()}).T

,coef,std. error,t
Classical (iid),0.4479,0.0193,23.2361
Robust (HC1),0.4479,0.0210,21.3591
Clustered by village,0.4479,0.0229,19.5385


Here the inflation is mild — the effect is enormous relative to any of these standard errors, so
nothing turns on the choice. **Do not generalize from that.** In §5 you will see a design where
clustering changes the effective sample size by a factor of fifty. The reason it is mild here is
that the treatment varies *within* village; when treatment is assigned *to* whole clusters, the
picture changes completely.

### Exercise 2 (5 min) — dose response

`any` throws away information: `tinc` records the actual incentive amount, and Thornton
randomized the amount too. Estimate the effect of the incentive *amount* on `got`, with village
clusters. Then add `any` and `tinc` to the same regression.

Interpret both coefficients in the second specification. What is the economic content of the
gap between the effect of "getting any money at all" and the effect of "getting one more dollar"?

In [41]:
# ---- EXERCISE 2 -------------------------------------------------------------
cl = {"groups": d.villnum}

dose  = smf.ols("got ~ tinc", data=d).fit(cov_type="cluster", cov_kwds=cl)        # got ~ tinc, clustered
both  = smf.ols("got ~ any + tinc", data=d).fit(cov_type="cluster", cov_kwds=cl)  # got ~ any + tinc, clustered

for name, m in [("got ~ tinc", dose), ("got ~ any + tinc", both)]:
    print(name)
    print(m.summary().tables[1])
    print()
# -----------------------------------------------------------------------------


got ~ tinc
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5129      0.021     24.882      0.000       0.472       0.553
tinc           0.1770      0.011     16.033      0.000       0.155       0.199

got ~ any + tinc
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.3414      0.024     14.380      0.000       0.295       0.388
any            0.3394      0.027     12.741      0.000       0.287       0.392
tinc           0.0842      0.012      6.948      0.000       0.060       0.108



**What to notice:** almost all of the action is in `any`, not in `tinc`. Going from zero to *any*
money moves behavior enormously; going from a small amount to a larger amount moves it much less.
That pattern is hard to reconcile with a pure "cost of travel" story and much easier to reconcile
with something like a salience or reminder effect. Thornton discusses exactly this. It is also a
nice illustration of a point from Thursday: **treatment is a bundle**, and a single binary
indicator can hide the mechanism you actually care about.



## Takeaways

1. Clean before you estimate. Out-of-range codes and non-random missingness change your estimand
   before they change your standard errors.
2. Difference in means, Welch $t$-test, and robust OLS are the same estimate. Use the regression,
   because it extends.
3. Cluster at the level of assignment. When treatment is assigned to clusters, the number of
   clusters is your real sample size.

